In [ ]:
import numpy as np

from theia.coordinates import CoordinateTransformations
from theia.measurement import MonostaticMeasurementTransformations
from theia.types import Point


p_observer = Point(lat=47.3495, lon=8.4921, alt=856.2)
p_observer_ecef = np.asarray(
    CoordinateTransformations.geodetic_to_cartesian(*p_observer.as_tuple())
)

p_target = Point(lat=47.34979, lon=8.40729, alt=1000)
p_target_ecef = np.asarray(
    CoordinateTransformations.geodetic_to_cartesian(*p_target.as_tuple())
)

result = MonostaticMeasurementTransformations.cartesian_to_elevation_azimuth_range(
    p_observer, p_target_ecef
)

In [ ]:
%%timeit
MonostaticMeasurementTransformations.cartesian_to_elevation_azimuth_range(
    p_observer, p_target_ecef
)

In [ ]:
import cProfile


profiler = cProfile.Profile()

profiler.enable()

for _ in range(100_000):
    MonostaticMeasurementTransformations.cartesian_to_elevation_azimuth_range(
        p_observer, p_target_ecef
    )

profiler.disable()
profiler.dump_stats("profile__cartesian_to_elevation_azimuth_range.prof")

In [ ]:
from theia.coordinates import calculate_elevation_angle, calculate_elevation_angle_ecef


calculate_elevation_angle(p_observer, p_target)
calculate_elevation_angle_ecef(p_observer_ecef, p_target_ecef)

In [ ]:
%%timeit
calculate_elevation_angle(p_observer, p_target)

In [ ]:
%%timeit
calculate_elevation_angle_ecef(p_observer_ecef, p_target_ecef)

In [ ]:
%%timeit
np.asarray(
    CoordinateTransformations.geodetic_to_cartesian(*p_observer.as_tuple())
)

In [ ]:
from theia.coordinates import _ecef_to_enu_rotation_matrix


def ecef_to_enu(
    reference_point: Point, p_ecef: tuple[float, float, float]
) -> tuple[float, float, float]:
    """
    Convert Cartesian coordinates from earth-centered-earth-fixed to east-north-up.

    Parameters
    ----------
    reference_point: Point
        Reference point in geodetic coordinates at which the
        ENU-frame is defined. Typically the observer (radar) position.
    p_ecef: tuple[float, float, float]
        Point in Cartesian ECEF coordinates to be transformed to ENU
        coordinates. Typically the observed (target) position.

    Returns
    -------
    tuple[float, float, float]
        Cartesian ENU coordinates corresponding to p_ecef

    Notes
    -----
    Formula according to Wikipedia:
    https://en.wikipedia.org/wiki/Geographic_coordinate_conversion#From_ECEF_to_ENU
    """
    R_ecef_to_enu = _ecef_to_enu_rotation_matrix(
        reference_point.lat,
        reference_point.lon,
    )

    reference_point_xyz = np.array(
        CoordinateTransformations.geodetic_to_cartesian(
            *reference_point.as_tuple(),
        )
    )
    p_ecef = np.array(p_ecef)
    p_enu = R_ecef_to_enu @ (p_ecef - reference_point_xyz)
    return (float(p_enu[0]), float(p_enu[1]), float(p_enu[2]))


def ecef_to_enu_new(
    reference_point_ecef: tuple[float, float, float], p_ecef: tuple[float, float, float]
) -> tuple[float, float, float]:
    """
    Convert Cartesian coordinates from earth-centered-earth-fixed to east-north-up.

    Parameters
    ----------
    reference_point: Point
        Reference point in geodetic coordinates at which the
        ENU-frame is defined. Typically the observer (radar) position.
    p_ecef: tuple[float, float, float]
        Point in Cartesian ECEF coordinates to be transformed to ENU
        coordinates. Typically the observed (target) position.

    Returns
    -------
    tuple[float, float, float]
        Cartesian ENU coordinates corresponding to p_ecef

    Notes
    -----
    Formula according to Wikipedia:
    https://en.wikipedia.org/wiki/Geographic_coordinate_conversion#From_ECEF_to_ENU
    """
    R_ecef_to_enu = _ecef_to_enu_rotation_matrix(
        reference_point.lat,
        reference_point.lon,
    )

    reference_point_xyz = np.array(
        CoordinateTransformations.geodetic_to_cartesian(
            *reference_point.as_tuple(),
        )
    )
    p_ecef = np.array(p_ecef)
    p_enu = R_ecef_to_enu @ (p_ecef - reference_point_xyz)
    return (float(p_enu[0]), float(p_enu[1]), float(p_enu[2]))


def enu_to_ecef(
    reference_point: Point, p_enu: tuple[float, float, float]
) -> tuple[float, float, float]:
    """
    Convert Cartesian coordinates from east-north-up to earth-centered-earth-fixed.

    Parameters
    ----------
    reference_point: Point
        Reference point in geodetic coordinates at which the
        ENU-frame is defined. Typically the observer (radar) position.
    p_enu: tuple[float, float, float]
        Point in Cartesian ENU coordinates to be transformed to ECEF
        coordinates. Typically the observed (target) position.

    Returns
    -------
    tuple[float, float, float]
        Cartesian ECEF coordinates corresponding to p_enu

    Notes
    -----
    Formula according to Wikipedia:
    https://en.wikipedia.org/wiki/Geographic_coordinate_conversion#From_ENU_to_ECEF
    """
    R_ecef_to_enu = _ecef_to_enu_rotation_matrix(
        reference_point.lat,
        reference_point.lon,
    )
    R_enu_to_ecef = R_ecef_to_enu.T

    reference_point_xyz = np.array(
        CoordinateTransformations.geodetic_to_cartesian(
            *reference_point.as_tuple(),
        )
    )
    p_enu = np.array(p_enu)
    p_ecef = R_enu_to_ecef @ p_enu + reference_point_xyz
    return (float(p_ecef[0]), float(p_ecef[1]), float(p_ecef[2]))

In [ ]:
ecef_to_enu(p_observer, p_target_ecef)

In [ ]:
%%timeit
ecef_to_enu(p_observer, p_target_ecef)

In [ ]:
import numba


def _ecef_to_enu_rotation_matrix(lat: float, lon: float) -> np.array:
    """
    Calculate rotation matrix that converts earth-centered-earth-fixed to East-North-Up
    coordinates at the given lat, lon coordinates.

    Parameters
    ----------
    lat: float
        Latitude [°]
    lon: float
        Longitude [°]

    Returns
    -------
    np.ndarray
        Rotation matrix of shape (3, 3)
    """
    lon = np.deg2rad(lon)
    lat = np.deg2rad(lat)
    R_ecef_to_enu = np.array(
        [
            [-np.sin(lon), np.cos(lon), 0.0],
            [-np.sin(lat) * np.cos(lon), -np.sin(lat) * np.sin(lon), np.cos(lat)],
            [np.cos(lat) * np.cos(lon), np.cos(lat) * np.sin(lon), np.sin(lat)],
        ]
    )
    return R_ecef_to_enu


@numba.njit
def _ecef_to_enu_rotation_matrix_new(lat: float, lon: float) -> np.array:
    """
    Calculate rotation matrix that converts earth-centered-earth-fixed to East-North-Up
    coordinates at the given lat, lon coordinates.

    Parameters
    ----------
    lat: float
        Latitude [°]
    lon: float
        Longitude [°]

    Returns
    -------
    np.ndarray
        Rotation matrix of shape (3, 3)
    """
    lon = np.deg2rad(lon)
    lat = np.deg2rad(lat)
    R_ecef_to_enu = np.empty(shape=(3, 3), dtype=np.float32)
    R_ecef_to_enu[0, :] = (-np.sin(lon), np.cos(lon), 0.0)
    R_ecef_to_enu[1, :] = (-np.sin(lat) * np.cos(lon), -np.sin(lat) * np.sin(lon), np.cos(lat))
    R_ecef_to_enu[2, :] = (np.cos(lat) * np.cos(lon), np.cos(lat) * np.sin(lon), np.sin(lat))
    return R_ecef_to_enu

In [ ]:
_ecef_to_enu_rotation_matrix(p_observer.lat, p_observer.lon)

In [ ]:
%%timeit
_ecef_to_enu_rotation_matrix(p_observer.lat, p_observer.lon)

In [ ]:
_ecef_to_enu_rotation_matrix_new(p_observer.lat, p_observer.lon)

In [ ]:
%%timeit
_ecef_to_enu_rotation_matrix_new(p_observer.lat, p_observer.lon)